# Water Stress Watch v1 — ML Training Notebook

**Purpose:** Train the v1 XGBoost model that predicts per-facility water-use efficiency (WUE) from climate and operator features, replacing v0's flat `WUE=1.8` assumption.

**Author:** Hiroaki Oshima (interactive execution on Colab Pro A100)
**Generated by:** `src/build_ml_training_set.py` (training set) + `src/build_v1_features.py` (inference features)
**Locked decisions (Week 4):** see `docs/handoff_week_4.md`

## What this notebook does
1. Loads the training set (~60-100 rows from Google / Microsoft / Meta / AWS disclosures).
2. Loads the v1 inference features (1,575 US data centers from v0).
3. Engineers features: one-hot encoding, missing-value handling, train/test split.
4. **Validates** with stratified 5-fold k-fold (across operators) + leave-one-operator-out generalization test.
5. Trains an XGBoost model on WUE (L/kWh).
6. Reports RMSE / MAE / R² on validation, feature importance, and SHAP values.
7. Predicts WUE for all 1,575 v0 facilities and applies the v1 inference formula.
8. Saves the model artifact and predictions to Google Drive for download.

## What this notebook does NOT do
- Optuna hyperparameter sweep (Week 5+)
- Cooling-type classifier (Week 5+)
- Sub-basin (HUC-8) stress overlay (Week 5+)
- Replacing the v0 map with v1 output (deferred until v1 is validated)

## Run order
Run cells 1-13 sequentially. Cell 14 (training) should run on A100. Cell 15+ save outputs.



In [ ]:
# Cell 2: pip install
# Colab Pro A100 runtime has most of these, but pin versions for reproducibility.
!pip install --quiet \
    xgboost>=2.0.0 \
    scikit-learn>=1.4.0 \
    shap>=0.45.0 \
    optuna>=3.6.0 \
    joblib>=1.3.0 \
    pandas>=2.0.0 \
    numpy>=1.24.0 \
    matplotlib>=3.8.0
print("pip install done")



In [ ]:
# Cell 3: Imports
import os
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, KFold, StratifiedKFold, train_test_split

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("imports ok")



In [ ]:
# Cell 4: Mount Google Drive (for persistence)
# The model artifact and predictions will be saved here. After the run, download
# them to /root/project/datacenter_water_stress/models/ locally.
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/water_stress_watch_v1")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Drive mounted at {DRIVE_DIR}")



In [ ]:
# Cell 5: Load training set
# Hiroaki: upload data/processed/ml_training_set.csv to Drive first, OR clone
# the repo and sync from there. The script that produced this file is
# src/build_ml_training_set.py (idempotent).
TRAINING_CSV = DRIVE_DIR / "ml_training_set.csv"
if not TRAINING_CSV.exists():
    # Fallback: assume the CSV is at the repo root
    TRAINING_CSV = Path("ml_training_set.csv")

train_df = pd.read_csv(TRAINING_CSV)
print(f"Loaded {len(train_df)} training rows from {TRAINING_CSV}")
print(f"  per-operator: {train_df['operator'].value_counts().to_dict()}")
print(f"  with WUE: {train_df['wue_disclosed'].notna().sum()}")
print(f"  aggregate (is_aggregate=True): {train_df['is_aggregate'].sum()}")



In [ ]:
# Cell 6: Load v1 inference features (1,575 facilities to score)
INFERENCE_CSV = DRIVE_DIR / "v1_inference_features.csv"
if not INFERENCE_CSV.exists():
    INFERENCE_CSV = Path("v1_inference_features.csv")

inference_df = pd.read_csv(INFERENCE_CSV)
print(f"Loaded {len(inference_df)} inference rows from {INFERENCE_CSV}")
print(f"  columns: {list(inference_df.columns)}")



In [ ]:
# Cell 7: Feature engineering
# Drop the row with NaN WUE (Meta Mesa AZ — held out as a real test, not training).
train_clean = train_df.dropna(subset=["wue_disclosed"]).copy()
y = train_clean["wue_disclosed"].values
groups = train_clean["operator"].values  # for GroupKFold / leave-one-operator-out

# One-hot encode operator + cooling type.
X = pd.get_dummies(
    train_clean[["cooling_type", "operator", "is_aggregate"]].astype(str),
    drop_first=False,
)
# Add continuous features that the model should learn from.
X["latitude"] = train_clean["latitude"]
X["longitude"] = train_clean["longitude"]
X["report_year"] = train_clean["report_year"]
# Note: we're NOT using bws_score as a feature here (it would leak the v0
# stress signal back into the model — a v0.5 design-day wet-bulb should fix
# the underlying climate_adj issue, and then the v1 model can use bws).

print(f"Training matrix shape: {X.shape}")
print(f"Target mean: {y.mean():.3f} L/kWh, std: {y.std():.3f} L/kWh")
print(f"\nFeature columns: {list(X.columns)}")



In [ ]:
# Cell 8: Baseline — v0 physics model on the same training set
# The v0 formula is WUE=1.8*0.7=1.26 L/kWh (a flat constant). The v1 model
# must beat this RMSE on the training set.
V0_WUE_FLOOR = 1.8 * 0.7
y_baseline = np.full_like(y, V0_WUE_FLOOR, dtype=float)
rmse_baseline = np.sqrt(mean_squared_error(y, y_baseline))
mae_baseline = mean_absolute_error(y, y_baseline)
r2_baseline = r2_score(y, y_baseline)
print(f"v0 baseline (WUE=1.26 constant):")
print(f"  RMSE = {rmse_baseline:.3f} L/kWh")
print(f"  MAE  = {mae_baseline:.3f} L/kWh")
print(f"  R^2  = {r2_baseline:.3f}  (negative because the model is a flat constant)")
print(f"\nNote: a flat-constant baseline will have a NEGATIVE R^2 because it can't")
print(f"capture operator / climate variance. v1 must beat this on all three metrics.")



In [ ]:
# Cell 9: XGBoost model (default hyperparameters)
# TODO: run with A100. Default hyperparameters are a starting point; Hiroaki
# will tune with Optuna in Week 5+.

xgb_params = {
    "n_estimators": 500,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": RANDOM_STATE,
    "tree_method": "hist",   # works on A100
    "device": "cuda",        # A100 GPU acceleration
}
model = xgb.XGBRegressor(**xgb_params)
print(f"XGBoost model: {xgb_params}")
print(f"\nNote: don't call model.fit() yet — we want to do CV first to see")
print(f"how the default hyperparams perform before committing to a full fit.")



In [ ]:
# Cell 10: Validation
# 10a: Stratified 5-fold k-fold across all 4 operators.
#      Stratify by operator so each fold has roughly equal operator mix.
# 10b: Leave-one-operator-out — train on 3, test on the held-out 4th.
#      This is the "how well does the model generalize" test.

print("=== 10a: Stratified 5-fold k-fold ===")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
fold_metrics = []
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, groups), 1):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    m = xgb.XGBRegressor(**xgb_params)
    m.fit(X_tr, y_tr, verbose=False)
    pred = m.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    mae = mean_absolute_error(y_val, pred)
    r2 = r2_score(y_val, pred)
    fold_metrics.append({"fold": fold, "rmse": rmse, "mae": mae, "r2": r2})
    print(f"  fold {fold}: RMSE={rmse:.3f}  MAE={mae:.3f}  R^2={r2:.3f}")
mean_rmse = np.mean([f["rmse"] for f in fold_metrics])
print(f"  mean RMSE across 5 folds: {mean_rmse:.3f} L/kWh  (must beat baseline {rmse_baseline:.3f})")

print("\n=== 10b: Leave-one-operator-out ===")
loo_operators = sorted(set(groups))
loo_metrics = []
for held_out_op in loo_operators:
    tr_mask = groups != held_out_op
    val_mask = groups == held_out_op
    if val_mask.sum() < 3:
        print(f"  skip {held_out_op}: only {val_mask.sum()} rows")
        continue
    m = xgb.XGBRegressor(**xgb_params)
    m.fit(X[tr_mask], y[tr_mask], verbose=False)
    pred = m.predict(X[val_mask])
    rmse = np.sqrt(mean_squared_error(y[val_mask], pred))
    mae = mean_absolute_error(y[val_mask], pred)
    r2 = r2_score(y[val_mask], pred)
    loo_metrics.append({"operator": held_out_op, "rmse": rmse, "mae": mae, "r2": r2, "n": val_mask.sum()})
    print(f"  held-out {held_out_op:12s} (n={val_mask.sum():2d}): "
          f"RMSE={rmse:.3f}  MAE={mae:.3f}  R^2={r2:.3f}")
print("\nLOO is the strictest test. If R^2 is much lower than 5-fold, the model")
print("is overfitting to operator-specific patterns. If it's similar, it generalizes.")



In [ ]:
# Cell 11: Feature importance (XGBoost built-in)
# Fit on the full training set for the importance plot.
model_full = xgb.XGBRegressor(**xgb_params)
model_full.fit(X, y, verbose=False)
importance = pd.Series(model_full.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 15 features by gain:")
print(importance.head(15).to_string())

fig, ax = plt.subplots(figsize=(10, 6))
importance.head(15).plot(kind="barh", ax=ax)
ax.set_xlabel("Feature importance (gain)")
ax.set_title("v1 XGBoost — top 15 features")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(DRIVE_DIR / "v1_feature_importance.png", dpi=120)
plt.show()



In [ ]:
# Cell 12: SHAP values
# SHAP gives per-prediction attribution. With ~60-100 training rows, the
# summary plot will be small but interpretable.
explainer = shap.TreeExplainer(model_full)
shap_values = explainer.shap_values(X)
shap.summary_plot(shap_values, X, show=False)
plt.tight_layout()
plt.savefig(DRIVE_DIR / "v1_shap_summary.png", dpi=120)
plt.show()
print("SHAP values saved to", DRIVE_DIR / "v1_shap_summary.png")



In [ ]:
# Cell 13: Predict WUE for all 1,575 v0 facilities
# Build the inference feature matrix the same way as the training matrix.
# The model only knows the columns it was trained on, so we must align.

# Reuse the same one-hot encoding as the training matrix.
inference_X = pd.get_dummies(
    inference_df[["cooling_type", "operator", "is_aggregate"]].astype(str),
    drop_first=False,
)
inference_X["latitude"] = inference_df["latitude"]
inference_X["longitude"] = inference_df["longitude"]
inference_X["report_year"] = 2024  # constant for v1; future re-trains update

# Align columns with the training matrix (any missing -> 0).
inference_X = inference_X.reindex(columns=X.columns, fill_value=0)
print(f"Inference matrix shape: {inference_X.shape}  (expected: {(len(inference_df), X.shape[1])})")

# Predict.
v1_predicted_wue = model_full.predict(inference_X)
inference_df["v1_predicted_wue"] = v1_predicted_wue
print(f"\nPredicted WUE stats:")
print(f"  min    = {v1_predicted_wue.min():.3f} L/kWh")
print(f"  median = {np.median(v1_predicted_wue):.3f} L/kWh")
print(f"  max    = {v1_predicted_wue.max():.3f} L/kWh")
print(f"  mean   = {v1_predicted_wue.mean():.3f} L/kWh")



In [ ]:
# Cell 14: Save model artifact + predictions
# Download to local: /root/project/datacenter_water_stress/models/

# 14a: model artifact
model_path = DRIVE_DIR / "water_estimator_v1.pkl"
joblib.dump(model_full, model_path)
print(f"Model saved to {model_path}")

# 14b: per-facility WUE predictions
preds_path = DRIVE_DIR / "v1_predicted_wue.csv"
out_cols = [
    "dc_id", "name", "provider", "state", "latitude", "longitude",
    "operator_class", "is_hyperscaler_self", "is_colocation_major",
    "is_colocation_secondary", "est_mw", "wet_bulb_c", "climate_adj",
    "est_liters_per_day", "v1_predicted_wue", "bws_score", "bws_category",
]
inference_df[out_cols].to_csv(preds_path, index=False)
print(f"Predictions saved to {preds_path}")

# 14c: the v1 inference features CSV (so Hiroaki has it for the next session)
inference_path = DRIVE_DIR / "v1_inference_features.csv"
inference_df.to_csv(inference_path, index=False)
print(f"v1 features saved to {inference_path}")
print("\nAfter the run, download these three files to:")
print("  /root/project/datacenter_water_stress/models/water_estimator_v1.pkl")
print("  /root/project/datacenter_water_stress/data/processed/v1_predicted_wue.csv")
print("  /root/project/datacenter_water_stress/data/processed/v1_inference_features.csv")



In [ ]:
# Cell 15: Comparison plot — v0 physics WUE=1.8 vs v1 ML by state
# Aggregate to state level for the comparison.
v1_lpd_per_row = (
    inference_df["est_mw"] * 1000 * 24 * 0.7 * inference_df["v1_predicted_wue"]
    * inference_df["climate_adj"]
)
inference_df["v1_liters_per_day"] = v1_lpd_per_row

state_compare = (
    inference_df.dropna(subset=["est_mw"])
    .groupby("state")
    .agg(
        n=("dc_id", "count"),
        total_mw=("est_mw", "sum"),
        v0_total_lpd=("est_liters_per_day", "sum"),
        v1_total_lpd=("v1_liters_per_day", "sum"),
    )
)
state_compare["v1_minus_v0_pct"] = (
    (state_compare["v1_total_lpd"] - state_compare["v0_total_lpd"])
    / state_compare["v0_total_lpd"] * 100
)
state_compare = state_compare.sort_values("v1_minus_v0_pct", key=abs, ascending=False)

print("Top 10 states by |v1 - v0| % difference:")
print(state_compare.head(10).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
state_compare.head(15).plot(
    kind="bar", y=["v0_total_lpd", "v1_total_lpd"], ax=axes[0], logy=True
)
axes[0].set_title("v0 vs v1 est. water L/day by state (log scale)")
axes[0].set_ylabel("L/day (log)")
state_compare.head(15).plot(
    kind="bar", y="v1_minus_v0_pct", ax=axes[1], color="darkred"
)
axes[1].set_title("v1 - v0 % difference (top 15 states)")
axes[1].axhline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.savefig(DRIVE_DIR / "v1_vs_v0_comparison.png", dpi=120)
plt.show()



In [ ]:
# Cell 16: v1 output schema (documentation)
# When the v1 model is applied to all 1,575 v0 facilities, the output for
# each facility is:

OUTPUT_SCHEMA = {
    "dc_id": "string (FK to v0 table)",
    "name": "string (facility name)",
    "state": "2-letter state code",
    "operator_class": "string (one of the v0 heuristic classes)",
    "is_hyperscaler_self": "bool — one-hot feature",
    "is_colocation_major": "bool — one-hot feature",
    "is_colocation_secondary": "bool — one-hot feature",
    "est_mw": "float — v0 best-estimate nameplate MW",
    "wet_bulb_c": "float — v0 annual mean wet-bulb (TODO: design-day in v0.5)",
    "climate_adj": "float — v0 climate adjustment multiplier",
    "est_liters_per_day": "float — v0 physics estimate (baseline; v1 must beat)",
    "v1_predicted_wue": "float — v1 XGBoost predicted WUE (L/kWh)",
    "v1_liters_per_day": "float — v1 estimate: est_mw * 1000 * 24 * 0.7 * v1_wue * climate_adj",
    "v1_minus_v0_pct": "float — (v1 - v0) / v0 * 100; the journalism headline",
    "bws_score": "float — WRI BWS for the state (analysis only, NOT a v1 feature)",
    "bws_category": "string — WRI category",
}

for k, v in OUTPUT_SCHEMA.items():
    print(f"  {k:25s} : {v}")

print("\n=== v1 model output saved. To merge back into v0: ===")
print("  pd.read_csv('data/processed/v1_predicted_wue.csv').merge(")
print("      pd.read_csv('data/processed/us_dc_with_stress.csv')[['dc_id']], on='dc_id'")
print("  )")

